In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torch.nn.functional as F
import os
import numpy as np
from torch.utils.data import Dataset
import json

In [3]:
!cp -r /content/drive/MyDrive/ML_Project/spectrogram_dataset.zip /content/


In [4]:
!unzip /content/spectrogram_dataset.zip -d /content/


Streaming output truncated to the last 5000 lines.
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05622_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05623_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05624_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05625_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05626_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05627_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05628_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_phase/sample_05629_snr0.npy  
  inflating: /content/content/drive/MyDrive/ML_Project/spectr

In [20]:
class SpectrogramDataset(Dataset):
    def __init__(self, clean_dir, noisy_dir, transform=None):
        """
        clean_dir: path to clean magnitude spectrograms
        noisy_dir: path to noisy magnitude spectrograms
        transform: optional function for normalization/augmentation
        """
        self.clean_dir = clean_dir
        self.noisy_dir = noisy_dir
        self.transform = transform

        # List of files (same names for noisy & clean)
        self.filenames = sorted(os.listdir(clean_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):

      filename = self.filenames[idx]

      clean = np.load(os.path.join(self.clean_dir, filename)).astype(np.float32)
      noisy = np.load(os.path.join(self.noisy_dir, filename)).astype(np.float32)

      clean = clean[:256, :]
      noisy = noisy[:256, :]


      clean = torch.from_numpy(clean).unsqueeze(0)
      noisy = torch.from_numpy(noisy).unsqueeze(0)

      return noisy, clean, filename



In [21]:
clean_dir = "/content/content/drive/MyDrive/ML_Project/spectrogram_dataset/clean_mag"
noisy_dir = "/content/content/drive/MyDrive/ML_Project/spectrogram_dataset/noisy_mag"

dataset = SpectrogramDataset(clean_dir, noisy_dir)

In [22]:
noisy, clean, name = dataset[0]
print(noisy.shape, clean.shape)
print(noisy.min(), noisy.max())

torch.Size([1, 256, 256]) torch.Size([1, 256, 256])
tensor(0.) tensor(5.5426)


In [23]:
from torch.utils.data import DataLoader, random_split

# Train/val split (90/10)
total = len(dataset)
train_size = int(0.9 * total)
val_size = total - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [24]:
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        # Resize for matching shapes
        #if g1.size()[2:] != x1.size()[2:]:
        #    x1 = F.interpolate(x1, size=g1.shape[2:], mode="bilinear", align_corners=False)

        psi = self.relu(g1 + x1)
        psi = self.psi(psi)

        # Resize psi to match skip connection
        #if psi.size()[2:] != x.size()[2:]:
        #    psi = F.interpolate(psi, size=x.shape[2:], mode="bilinear", align_corners=False)

        return x * psi


In [25]:

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


In [26]:
class AttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1):
        super(AttentionUNet, self).__init__()

        # Encoder
        self.conv1 = DoubleConv(in_ch, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.conv4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.conv5 = DoubleConv(512, 1024)

        # Attention Gates
        self.att4 = AttentionGate(F_g=512, F_l=512, F_int=256)
        self.att3 = AttentionGate(F_g=256, F_l=256, F_int=128)
        self.att2 = AttentionGate(F_g=128, F_l=128, F_int=64)
        self.att1 = AttentionGate(F_g=64,  F_l=64,  F_int=32)


        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Output
        self.out_conv = nn.Conv2d(64, out_ch, kernel_size=1)

    def forward(self, x):
        # ----- Encoder -----
        x1 = self.conv1(x)
        p1 = self.pool1(x1)

        x2 = self.conv2(p1)
        p2 = self.pool2(x2)

        x3 = self.conv3(p2)
        p3 = self.pool3(x3)

        x4 = self.conv4(p3)
        p4 = self.pool4(x4)

        x5 = self.conv5(p4)

        # ----- Decoder with Attention -----
        g4 = self.up4(x5)
        x4_att = self.att4(g4, x4)
        #x4_att = F.interpolate(x4_att, size=g4.shape[2:], mode="bilinear", align_corners=False)
        d4 = self.dec4(torch.cat([g4, x4_att], dim=1))

        g3 = self.up3(d4)
        x3_att = self.att3(g3, x3)
        #x3_att = F.interpolate(x3_att, size=g3.shape[2:], mode="bilinear", align_corners=False)
        d3 = self.dec3(torch.cat([g3, x3_att], dim=1))

        g2 = self.up2(d3)
        x2_att = self.att2(g2, x2)
        #x2_att = F.interpolate(x2_att, size=g2.shape[2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([g2, x2_att], dim=1))

        g1 = self.up1(d2)
        x1_att = self.att1(g1, x1)
        #x1_att = F.interpolate(x1_att, size=g1.shape[2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([g1, x1_att], dim=1))

        return self.out_conv(d1)


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AttentionUNet().to(device)

l1 = nn.L1Loss()
l2 = nn.MSELoss()

def criterion(pred, target):
    return 0.8 * l1(pred, target) + 0.2 * l2(pred, target)

optimizer = optim.Adam(model.parameters(), lr=3e-4)


In [28]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0

    for noisy, clean, _ in tqdm(loader):
        noisy = noisy.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()

        output = model(noisy)

        # Resize model output to match the clean spectrogram
        #if output.shape[2:] != clean.shape[2:]:
        #    output = F.interpolate(output, size=clean.shape[2:], mode="bilinear", align_corners=False)

        loss = criterion(output, clean)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0

    with torch.no_grad():
        for noisy, clean, _ in loader:
            noisy = noisy.to(device)
            clean = clean.to(device)

            output = model(noisy)
            # Resize model output to match the clean spectrogram
            #if output.shape[2:] != clean.shape[2:]:
            #    output = F.interpolate(output, size=clean.shape[2:], mode="bilinear", align_corners=False)

            loss = criterion(output, clean)

            running_loss += loss.item()

    return running_loss / len(loader)


In [29]:
save_dir = "/content/drive/MyDrive/ML_Project/Model2"
os.makedirs(save_dir, exist_ok=True)


In [30]:
import os
import re
import torch

save_dir = "/content/drive/MyDrive/ML_Project/Model2"

# Find all saved checkpoints
saved_epochs = []

for fname in os.listdir(save_dir):
    match = re.match(r"checkpoint_epoch_(\d+)\.pth", fname)
    if match:
        saved_epochs.append(int(match.group(1)))

# Determine starting point
if len(saved_epochs) > 0:
    last_epoch = max(saved_epochs)
    checkpoint_path = f"{save_dir}/checkpoint_epoch_{last_epoch}.pth"

    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    start_epoch = checkpoint["epoch"]
    best_val_loss = checkpoint["best_val_loss"]

    print(f"🔄 Loaded checkpoint: epoch {last_epoch}")
else:
    start_epoch = 0
    best_val_loss = float("inf")
    print("No checkpoint found — starting from epoch 0")


No checkpoint found — starting from epoch 0


In [ ]:
EPOCHS = 50
save_dir = "/content/drive/MyDrive/ML_Project/Model2"

train_losses = []
val_losses = []


for epoch in range(start_epoch, EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)


    print(f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = f"{save_dir}/best_model.pth"
        torch.save(model.state_dict(), best_path)
        print(f"✔ Best model updated and saved at: {best_path}")


    # Save model + optimizer at every epoch
    checkpoint = {
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
    }

    epoch_path = f"{save_dir}/checkpoint_epoch_{epoch+1}.pth"
    torch.save(checkpoint, epoch_path)
    print (f" Epoch checkpoint saved at: {epoch_path}")

    # Save loss history every epoch
    with open("/content/drive/MyDrive/ML_Project/train_losses.jsonl", "a") as f:
        f.write(json.dumps({
        "epoch": epoch + 1,
        "loss": train_loss
    }) + "\n")

    with open("/content/drive/MyDrive/ML_Project/val_losses.jsonl", "a") as f:
         f.write(json.dumps({
        "epoch": epoch + 1,
        "loss": val_loss
    }) + "\n")


Epoch 1/50


100%|██████████| 2117/2117 [19:28<00:00,  1.81it/s]


Train Loss: 0.250582 | Val Loss: 0.220185
✔ Best model updated and saved at: /content/drive/MyDrive/ML_Project/Model2/best_model.pth
 Epoch checkpoint saved at: /content/drive/MyDrive/ML_Project/Model2/checkpoint_epoch_1.pth

Epoch 2/50


 89%|████████▉ | 1881/2117 [17:17<02:09,  1.82it/s]